In [6]:
!pip install faker numpy pandas

In [7]:
# notebooks/01_generate_synthetic_data.ipynb

import os
import pandas as pd
from faker import Faker
import numpy as np
import sys
from pathlib import Path

# Make parent directory importable so config.py can be imported
root = Path.cwd().parent
sys.path.insert(0, str(root))
from config import DATA_DIR_RAW

In [8]:
fake = Faker()
np.random.seed(42)
Faker.seed(42)

# Use centralized DATA_DIR from package config
# DATA_DIR and DATA_DIR_RAW are provided by supply_chain_optimization_kg.config
os.makedirs(DATA_DIR_RAW, exist_ok=True)

# ---------- Config ----------
num_suppliers = 10
num_products = 20
num_warehouses = 5
num_customers = 15
num_routes = 8
num_orders = 50
num_inventory_rows = 40
num_shipments = 50
num_events = 20

In [9]:
# ---------- Suppliers ----------
suppliers_df = pd.DataFrame({
    "SupplierID": range(1, num_suppliers + 1),
    "Name": [fake.company() for _ in range(num_suppliers)],
    "Country": [fake.country_code() for _ in range(num_suppliers)],
    "SupplierType": np.random.choice(["Manufacturer", "Distributor", "Wholesaler"], size=num_suppliers, p=[0.4, 0.4, 0.2]),
    "ContractType": np.random.choice(["Standard", "Preferred", "Spot"], size=num_suppliers, p=[0.6, 0.3, 0.1]),
    "ReliabilityScore": np.round(np.random.uniform(4, 10, num_suppliers), 2),
    "LeadTimeDays": np.random.randint(10, 35, num_suppliers),
    "OnTimeDeliveryRate": np.round(np.random.uniform(0.72, 0.99, num_suppliers), 2),
    "QualityIssueCount": np.random.poisson(2, num_suppliers),
    "RiskRating": np.random.choice(["Low", "Medium", "High"], size=num_suppliers, p=[0.5, 0.35, 0.15]),
    "CapacityTier": np.random.choice(["Low", "Medium", "High"], size=num_suppliers, p=[0.4, 0.4, 0.2]),
})

# Add performance labels to support KG queries about slow or unreliable suppliers
suppliers_df["PerformanceCategory"] = suppliers_df.apply(
    lambda row: "Poor" if row["LeadTimeDays"] > 20 and row["ReliabilityScore"] < 7.0 else (
        "Excellent" if row["LeadTimeDays"] <= 15 and row["ReliabilityScore"] >= 8.5 else "Moderate"
    ),
    axis=1,
)

# Add a derived exposure score to support prioritisation analysis
suppliers_df["ExposureScore"] = np.round(
    100 - (
        (10 - suppliers_df["ReliabilityScore"]) * 6
        + (100 - suppliers_df["OnTimeDeliveryRate"] * 100) * 0.45
        + suppliers_df["QualityIssueCount"] * 4
    ),
    2,
)
suppliers_df["ExposureScore"] = suppliers_df["ExposureScore"].clip(lower=0, upper=100)

# ---------- Products ----------
products_df = pd.DataFrame({
    "ProductID": range(1, num_products + 1),
    "SupplierID": np.random.choice(suppliers_df["SupplierID"], size=num_products),
    "SKU": [str(fake.ean13())[-8:] for _ in range(num_products)],
    "Category": np.random.choice(["Electronics", "Clothing", "Food"], size=num_products),
    "DemandLevel": np.round(np.random.uniform(10, 100, num_products), 2),
    "StockQty": np.random.randint(20, 200, num_products)
})

# ---------- Warehouses ----------
capacities = np.random.randint(800, 2500, num_warehouses)
current_loads = [np.random.randint(int(c * 0.4), int(c * 0.9)) for c in capacities]

warehouses_df = pd.DataFrame({
    "WarehouseID": range(1, num_warehouses + 1),
    "Location": [fake.city() for _ in range(num_warehouses)],
    "Capacity": capacities,
    "CurrentLoad": current_loads
})

# ---------- Customers ----------
customers_df = pd.DataFrame({
    "CustomerID": range(1, num_customers + 1),
    "Region": np.random.choice(["North", "South", "East", "West"], size=num_customers),
    "Tier": np.random.randint(1, 5, num_customers)
})

# ---------- Routes ----------
route_rows = []
for rid in range(1, num_routes + 1):
    origin, destination = np.random.choice(warehouses_df["WarehouseID"], size=2, replace=False)
    transit_days = np.random.randint(2, 15)
    cost = np.round(np.random.uniform(50, 400) + transit_days * 10, 2)
    route_rows.append({
        "RouteID": rid,
        "OriginWarehouseID": origin,
        "DestinationWarehouseID": destination,
        "AvgTransitDays": transit_days,
        "Cost": cost
    })

routes_df = pd.DataFrame(route_rows)

# ---------- Orders ----------
order_dates = pd.to_datetime(
    np.random.choice(pd.date_range("2023-01-01", "2023-12-31", freq="D"), size=num_orders)
)

orders_df = pd.DataFrame({
    "OrderID": range(1, num_orders + 1),
    "CustomerID": np.random.choice(customers_df["CustomerID"], size=num_orders),
    "ProductID": np.random.choice(products_df["ProductID"], size=num_orders),
    "WarehouseID": np.random.choice(warehouses_df["WarehouseID"], size=num_orders),
    "Quantity": np.random.randint(1, 20, num_orders),
    "Status": np.random.choice(["Pending", "Shipped", "Delivered"], size=num_orders, p=[0.2, 0.4, 0.4]),
    "Priority": np.random.randint(1, 5, num_orders),
    "Date": order_dates
}).sort_values("Date").reset_index(drop=True)

# ---------- Inventory ----------
inventory_rows = []
for iid in range(1, num_inventory_rows + 1):
    warehouse_id = np.random.choice(warehouses_df["WarehouseID"])
    product_id = np.random.choice(products_df["ProductID"])
    stock_on_hand = np.random.randint(10, 300)
    reserved_qty = np.random.randint(0, max(1, int(stock_on_hand * 0.4)))
    reorder_point = np.random.randint(5, max(6, int(stock_on_hand * 0.3)))
    inventory_rows.append({
        "InventoryID": iid,
        "WarehouseID": warehouse_id,
        "ProductID": product_id,
        "StockOnHand": stock_on_hand,
        "ReservedQty": reserved_qty,
        "ReorderPoint": reorder_point
    })

inventory_df = pd.DataFrame(inventory_rows)

# ---------- Shipments ----------
shipment_rows = []
for sid in range(1, num_shipments + 1):
    order_row = orders_df.iloc[sid - 1]
    route_row = routes_df.sample(1).iloc[0]

    shipped_date = pd.Timestamp(order_row["Date"]) + pd.Timedelta(days=np.random.randint(0, 5))
    expected_delivery = shipped_date + pd.Timedelta(days=int(route_row["AvgTransitDays"]))
    late_days = np.random.choice([0, 0, 0, 1, 2, 3])
    actual_delivery = expected_delivery + pd.Timedelta(days=late_days)

    if late_days == 0:
        status = "On Time"
    elif late_days <= 2:
        status = "Delayed"
    else:
        status = "Late"

    shipment_rows.append({
        "ShipmentID": sid,
        "OrderID": int(order_row["OrderID"]),
        "RouteID": int(route_row["RouteID"]),
        "FromWarehouseID": int(route_row["OriginWarehouseID"]),
        "ToWarehouseID": int(route_row["DestinationWarehouseID"]),
        "ShippedDate": shipped_date.date(),
        "ExpectedDeliveryDate": expected_delivery.date(),
        "ActualDeliveryDate": actual_delivery.date(),
        "ShipmentStatus": status
    })

shipments_df = pd.DataFrame(shipment_rows)

# ---------- Logistics Events ----------
event_types = ["Delay", "Outage", "Damage", "Inspection", "Shipment Hold"]
event_rows = []

for eid in range(1, num_events + 1):
    etype = np.random.choice(event_types)
    supplier_id = np.random.choice(suppliers_df["SupplierID"]) if etype in ["Delay", "Outage"] else None
    route_id = np.random.choice(routes_df["RouteID"]) if etype in ["Delay", "Shipment Hold"] else None
    warehouse_id = np.random.choice(warehouses_df["WarehouseID"]) if etype in ["Damage", "Inspection"] else None
    order_id = np.random.choice(orders_df["OrderID"]) if etype in ["Shipment Hold", "Damage"] else None
    shipment_id = np.random.choice(shipments_df["ShipmentID"]) if etype in ["Delay", "Shipment Hold", "Damage"] else None

    event_rows.append({
        "EventID": eid,
        "EventType": etype,
        "DelayDays": np.random.randint(1, 15) if etype == "Delay" else 0,
        "Cause": fake.sentence(nb_words=6),
        "SupplierID": supplier_id,
        "RouteID": route_id,
        "WarehouseID": warehouse_id,
        "OrderID": order_id,
        "ShipmentID": shipment_id
    })

logistics_events_df = pd.DataFrame(event_rows)

In [10]:
# ---------- Save CSVs ----------
suppliers_df.to_csv(DATA_DIR_RAW / "suppliers.csv", index=False)
products_df.to_csv(DATA_DIR_RAW / "products.csv", index=False)
warehouses_df.to_csv(DATA_DIR_RAW / "warehouses.csv", index=False)
customers_df.to_csv(DATA_DIR_RAW / "customers.csv", index=False)
routes_df.to_csv(DATA_DIR_RAW / "routes.csv", index=False)
orders_df.to_csv(DATA_DIR_RAW / "orders.csv", index=False)
inventory_df.to_csv(DATA_DIR_RAW / "inventory.csv", index=False)
shipments_df.to_csv(DATA_DIR_RAW / "shipments.csv", index=False)
logistics_events_df.to_csv(DATA_DIR_RAW / "logistics_events.csv", index=False)

print("Final synthetic supply-chain dataset generated successfully.")

Final synthetic supply-chain dataset generated successfully.
